# BigPipe.Streams and Flows
**EN** — Build a windowed aggregation with BigPipe.Streams and query its state interactively; then deploy an in-broker Flow that filters, masks and routes records.

**ID** — Bangun agregasi berjendela dengan BigPipe.Streams dan query state-nya secara interaktif; lalu deploy Flow di dalam broker yang memfilter, menyamarkan, dan merutekan record.

*Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

In [ ]:
// Using a local build instead of nuget.org? Uncomment and point to <repo>/artifacts/nuget (absolute path):
// #i "nuget: C:/src/bigpipe/artifacts/nuget"
#r "nuget: BigPipe.Streams, 0.1.0"
#r "nuget: BigPipe.Analytics, 0.1.0"
using BigPipe.Client;
using BigPipe.Client.Admin;
using BigPipe.Analytics;
var bootstrap = Environment.GetEnvironmentVariable("BIGPIPE_BOOTSTRAP") ?? "localhost:9092";
var admin = new BigPipeAdminClient(Environment.GetEnvironmentVariable("BIGPIPE_ADMIN") ?? "http://localhost:9644");
var http = new BigPipeHttpClient(Environment.GetEnvironmentVariable("BIGPIPE_HTTP") ?? "http://localhost:8082");
string Unique(string p) => $"nb-{p}-{Guid.NewGuid().ToString("N")[..6]}";
var cluster = await admin.GetClusterAsync();
Console.WriteLine($"Connected to {cluster.ClusterId} (BigPipe {cluster.Version}) — {cluster.Credit}");

In [ ]:
using BigPipe.Streams;
public record Payment(string MerchantId, decimal Amount, string Status);
public record MerchantTotals(long Count = 0, decimal Amount = 0);

var input = Unique("payments");
await admin.CreateTopicAsync(input, 3);
var topology = new StreamsBuilder();
topology.Stream<string, Payment>(input)
    .Filter((_, p) => p.Status == "SETTLED")
    .GroupBy((_, p) => p.MerchantId)
    .WindowedBy(TumblingWindow.Of(TimeSpan.FromMinutes(1)).Grace(TimeSpan.FromMinutes(2)))
    .Aggregate(() => new MerchantTotals(), (_, p, acc) => acc with { Count = acc.Count + 1, Amount = acc.Amount + p.Amount }, Stores.Persistent("totals"));
var app = new StreamsApp(topology, new StreamsConfig { ApplicationId = Unique("agg"), Bootstrap = bootstrap });
await app.StartAsync();

await using (var p = new ProducerBuilder<string, Payment>().WithBootstrap(bootstrap).Build())
{
    var merchants = new[] { "Warung Kopi", "Toko Batik", "Bakmi Jogja" };
    var t0 = DateTimeOffset.UtcNow;
    await Task.WhenAll(Enumerable.Range(0, 300).Select(i => p.SendAsync(input, new Message<string, Payment>
    {
        Key = $"tx-{i}", Value = new Payment(merchants[i % 3], 10_000m * (1 + i % 7), i % 10 == 0 ? "PENDING" : "SETTLED"), Timestamp = t0.AddSeconds(i / 3.0),
    })));
}
var store = app.Store<Windowed<string>, MerchantTotals>("totals");
for (var k = 0; k < 100 && store.All().Sum(x => x.Value.Count) < 270; k++) await Task.Delay(200);
store.All().OrderBy(x => x.Key.Start).Select(x => new { window = $"{x.Key.StartTime:HH:mm}", merchant = x.Key.Key, x.Value.Count, amount = x.Value.Amount })

In [ ]:
await app.DisposeAsync();
var flow = Unique("big-payments");
await admin.DeployFlowAsync(new FlowSpec
{
    Name = flow, Input = input, Output = input + ".big",
    Processors = [FlowProcessor.FilterBy("this.amount > 50000"), FlowProcessor.MapWith("root = this\nroot.flagged = true")],
});
await Task.Delay(2000);
(await admin.ListFlowsAsync()).Where(f => f.Name == flow).Select(f => f.Stats)